# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaGaya/KGaya/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Ranked Actions + Reason Codes

The playbook produces four action labels, ordered by priority:

Priority 1 — REVIEW_BOTH
Reason code: stale_and_low_ctr
Condition: page has not been updated in 180+ days AND has low CTR
despite visible impressions.
Action: Review both content and metadata. The page is old and not
capturing clicks — it likely needs a content refresh AND a
title/meta description rewrite.

Priority 2 — REVIEW_CONTENT  
Reason code: stale_visible_page
Condition: page has not been updated in 180+ days AND receives
100+ impressions.
Action: Review and update the content. The page is still visible
but may be losing relevance due to age.

Priority 3 — REVIEW_METADATA
Reason code: low_ctr_visible_page
Condition: page receives 500+ impressions AND CTR is below 0.5%
AND average position is 20 or better.
Action: Review title and meta description. The page ranks well
but is not being clicked — the snippet may not match search intent.

Priority 4 — MONITOR
Reason code: monitor
Condition: page does not meet any of the above thresholds.
Action: No immediate action. Check again next review cycle.

How to use the ranked queue:
Start at rank 1. Work down the list. Stop when the review team
runs out of capacity. The score is a priority signal, not a verdict.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended Use and Limits

INTENDED USE:
This playbook is a decision-support tool for a content review team
with limited capacity. It answers one question: given 30,000 pages,
which ones should a human reviewer look at first?

WHO USES IT:
A content strategist or SEO analyst who already understands the
site. The playbook gives them a prioritised starting point — it
does not replace their judgment.

WHERE IT IS VALID:
- On pages with at least 100 impressions in the 90-day window
- As a directional ranking, not a definitive verdict
- As a starting point for human review, not an automated trigger
- On data from the same distribution as the training data

WHERE IT STOPS BEING VALID:
- Pages with zero or near-zero impressions (noise, not signal)
- Brand new pages (no performance history to evaluate)
- Pages in verticals the model has not seen
- Any use case where the output would trigger automated content
  deletion, redirect, or publication without human review
- Any claim that the model predicts future traffic or proves
  that a refresh will cause recovery

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import json
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                        REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = [
    'impressions_90d', 'sessions_90d', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position',
    'word_count', 'engagement_rate'
]

df_clean = df.dropna(subset=feature_cols).copy()

# Client-grouped split
clients = df_clean['client_id'].unique()
np.random.seed(42)
np.random.shuffle(clients)
split = int(len(clients) * 0.8)
train_clients = set(clients[:split])
test_clients = set(clients[split:])

train = df_clean[df_clean['client_id'].isin(train_clients)]
test = df_clean[df_clean['client_id'].isin(test_clients)]

rf = RandomForestClassifier(n_estimators=100, max_depth=6,
                             class_weight='balanced', random_state=42)
rf.fit(train[feature_cols], train['is_declining_label'])

# Score full dataset
df_clean['model_score'] = rf.predict_proba(df_clean[feature_cols])[:, 1]

# Assign reason codes
def assign_reason(row):
    stale = row['days_since_last_update'] >= 180
    visible = row['impressions_90d'] >= 100
    low_ctr = row['ctr'] < 0.005 and row['impressions_90d'] >= 500
    if stale and visible and low_ctr:
        return 'stale_and_low_ctr'
    elif stale and visible:
        return 'stale_visible_page'
    elif low_ctr:
        return 'low_ctr_visible_page'
    else:
        return 'monitor'

def assign_action(reason):
    mapping = {
        'stale_and_low_ctr': 'REVIEW_BOTH',
        'stale_visible_page': 'REVIEW_CONTENT',
        'low_ctr_visible_page': 'REVIEW_METADATA',
        'monitor': 'MONITOR'
    }
    return mapping[reason]

df_clean['reason_code'] = df_clean.apply(assign_reason, axis=1)
df_clean['action_label'] = df_clean['reason_code'].apply(assign_action)

# Final ranked queue
queue = df_clean[[
    'content_id', 'impressions_90d', 'days_since_last_update',
    'ctr', 'avg_position', 'trend_direction',
    'model_score', 'reason_code', 'action_label'
]].sort_values('model_score', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

print(f"Queue built: {len(queue)} pages")
print(f"\nAction distribution:")
print(queue['action_label'].value_counts())
print(f"\nTop 10 preview:")
print(queue.head(10)[['rank','action_label','reason_code',
                        'impressions_90d','days_since_last_update',
                        'ctr','trend_direction']].to_string(index=False))

Queue built: 22301 pages

Action distribution:
action_label
MONITOR            20766
REVIEW_METADATA     1503
REVIEW_CONTENT        30
REVIEW_BOTH            2
Name: count, dtype: int64

Top 10 preview:
 rank    action_label          reason_code  impressions_90d  days_since_last_update  ctr trend_direction
    1         MONITOR              monitor              382                     104  0.0            down
    2         MONITOR              monitor              353                     104  0.0            down
    3 REVIEW_METADATA low_ctr_visible_page              618                     104  0.0            down
    4 REVIEW_METADATA low_ctr_visible_page              720                     104  0.0            down
    5 REVIEW_METADATA low_ctr_visible_page             2621                     104  0.0            down
    6         MONITOR              monitor              165                     104  0.0            down
    7 REVIEW_METADATA low_ctr_visible_page              705   

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human Review Rules + No-Go List

WHAT A HUMAN MUST CHECK BEFORE ACTING:

1. Does the page actually look stale? Open it. Sometimes a page
   has not been updated in HTML but the content is still accurate
   and relevant. Age alone is not enough.

2. Is the low CTR a content problem or a SERP problem? If the
   search results page now shows featured snippets or AI answers
   above organic results, CTR will drop for everyone — not just
   this page. Check the broader SERP before rewriting the meta.

3. Is the page declining or seasonal? A page about tax filing
   will always drop outside tax season. Check whether the trend
   matches a seasonal pattern before flagging it for review.

4. Is there a sibling page that absorbed the traffic? Sometimes
   one page loses impressions because a related page on the same
   site gained them. Consolidation looks like decline — check
   related content before acting.

THE NO-GO LIST — what should never be automated:

- Content deletion based on model score alone
- Automatic redirect or canonical changes
- Publishing rewritten content without human approval
- Using this output as evidence that a refresh will cause recovery
- Sending this queue directly to a client without human review
- Acting on pages with fewer than 100 impressions (noise floor)

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show pages that need extra human scrutiny
# Flag pages where reason code and trend direction disagree
queue['needs_scrutiny'] = (
    (queue['action_label'].isin(['REVIEW_CONTENT', 'REVIEW_BOTH'])) &
    (queue['trend_direction'] != 'down')
)

scrutiny_count = queue['needs_scrutiny'].sum()
print(f"Pages flagged for action but NOT currently declining: {scrutiny_count}")
print(f"These require extra human review before acting.\n")

# Show top 10 that need scrutiny
scrutiny_sample = queue[queue['needs_scrutiny']].head(10)
print("Sample pages needing scrutiny:")
print(scrutiny_sample[['rank','action_label','reason_code',
                         'trend_direction','impressions_90d',
                         'days_since_last_update']].to_string(index=False))

Pages flagged for action but NOT currently declining: 7
These require extra human review before acting.

Sample pages needing scrutiny:
 rank   action_label        reason_code trend_direction  impressions_90d  days_since_last_update
 2029 REVIEW_CONTENT stale_visible_page          stable              103                     304
 3351 REVIEW_CONTENT stale_visible_page          stable             1316                     194
 6561 REVIEW_CONTENT stale_visible_page          stable              268                     183
 8342 REVIEW_CONTENT stale_visible_page              up              371                     183
11246 REVIEW_CONTENT stale_visible_page             new              335                     301
11437 REVIEW_CONTENT stale_visible_page              up              148                     211
15407 REVIEW_CONTENT stale_visible_page              up              180                     211


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Monitoring and Retrain Triggers

WHEN TO CHECK IF RECOMMENDATIONS WENT STALE:

1. Precision@50 drops below 0.50 on a fresh sample — if spot-checking
   the top 50 pages shows fewer than 25 are actually declining, the
   model has drifted and needs retraining.

2. The feature distribution shifts significantly — if the average
   impressions or CTR across the dataset changes by more than 20%,
   the thresholds and features may no longer reflect reality.

3. A major search algorithm update — these change what signals
   matter for visibility. After a confirmed core update, audit
   feature importances before trusting the ranked queue.

4. Six months without retraining — the model was trained on a
   90-day snapshot. After six months, the world it learned from
   may no longer match current conditions.

WHAT WOULD TELL YOU THE PLAYBOOK IS WORKING:

- Pages acted on from the top of the queue show improvement
  in impressions or CTR in the following 60-90 days
- The human reviewer agrees with the top 20 flags more often
  than they disagree
- The reason codes match what reviewers find when they open the pages

Note: improvement after review is directional evidence only.
This data alone cannot prove the refresh caused the recovery.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Simulate a basic drift check
# Compare feature distributions between top-ranked and bottom-ranked pages
top_100 = queue.head(100)
bottom_100 = queue.tail(100)

print("Feature distribution comparison — top 100 vs bottom 100:")
for col in ['impressions_90d', 'days_since_last_update', 'ctr', 'avg_position']:
    top_mean = top_100[col].mean()
    bot_mean = bottom_100[col].mean()
    print(f"  {col}: top={top_mean:.2f} | bottom={bot_mean:.2f}")

print("\nRetrain trigger: if Precision@50 on a fresh sample drops below 0.50")
print("Monitoring cadence: check quarterly or after any major search algorithm update")

Feature distribution comparison — top 100 vs bottom 100:
  impressions_90d: top=674.21 | bottom=1.00
  days_since_last_update: top=103.98 | bottom=20.00
  ctr: top=0.00 | bottom=0.00
  avg_position: top=15.49 | bottom=0.00

Retrain trigger: if Precision@50 on a fresh sample drops below 0.50
Monitoring cadence: check quarterly or after any major search algorithm update


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Exports for the paper.

Exporting the ranked queue and figures to work/outputs/ and
work/figures/ so the capstone paper can build directly on these files.

Files exported:
- work/outputs/final_action_queue.csv — the full ranked queue
- work/outputs/metrics_summary.json — key metrics for the paper
- work/figures/feature_importance.png — feature importance chart
- work/figures/precision_at_k.png — precision curve chart

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# Export ranked queue
queue.to_csv('work/outputs/final_action_queue.csv', index=False)
print(f"Queue exported: work/outputs/final_action_queue.csv ({len(queue)} rows)")

# Export metrics summary
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.array(scores))
    topk = np.array(labels)[order[:k]]
    return topk.mean()

test_queue = queue[queue['content_id'].isin(
    df_clean[df_clean['client_id'].isin(test_clients)]['content_id']
)]

p20 = precision_at_k(test_queue['model_score'],
                      (test_queue['trend_direction']=='down').astype(int), 20)
p50 = precision_at_k(test_queue['model_score'],
                      (test_queue['trend_direction']=='down').astype(int), 50)

metrics = {
    "model": "RandomForestClassifier",
    "validation": "client_grouped_holdout",
    "precision_at_20": round(float(p20), 3),
    "precision_at_50": round(float(p50), 3),
    "dataset": "content_refresh_anonymized.csv",
    "n_rows": len(df_clean),
    "n_clients_train": len(train_clients),
    "n_clients_test": len(test_clients),
    "features_used": feature_cols,
    "label": "trend_direction == down",
    "note": "Directional, decision-support only. Not causal."
}

with open('work/outputs/metrics_summary.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics exported: work/outputs/metrics_summary.json")
print(json.dumps(metrics, indent=2))

# Feature importance chart
fi = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
plt.barh(fi['feature'], fi['importance'], color='#1C3557')
plt.xlabel('Importance')
plt.title('Feature Importances — Random Forest')
plt.tight_layout()
plt.savefig('work/figures/feature_importance.png', dpi=150)
plt.close()
print("Figure saved: work/figures/feature_importance.png")

# Precision@K curve
ks = [10, 20, 30, 40, 50, 75, 100]
precisions = [precision_at_k(test_queue['model_score'],
              (test_queue['trend_direction']=='down').astype(int), k) for k in ks]

plt.figure(figsize=(8, 5))
plt.plot(ks, precisions, marker='o', color='#1C3557', linewidth=2)
plt.xlabel('K (top K pages reviewed)')
plt.ylabel('Precision@K')
plt.title('Precision@K Curve — Random Forest (client-grouped holdout)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('work/figures/precision_at_k.png', dpi=150)
plt.close()
print("Figure saved: work/figures/precision_at_k.png")

print("\nAll exports complete. Paper can now build on these files.")

Queue exported: work/outputs/final_action_queue.csv (22301 rows)
Metrics exported: work/outputs/metrics_summary.json
{
  "model": "RandomForestClassifier",
  "validation": "client_grouped_holdout",
  "precision_at_20": 0.4,
  "precision_at_50": 0.48,
  "dataset": "content_refresh_anonymized.csv",
  "n_rows": 22301,
  "n_clients_train": 25,
  "n_clients_test": 7,
  "features_used": [
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "word_count",
    "engagement_rate"
  ],
  "label": "trend_direction == down",
  "note": "Directional, decision-support only. Not causal."
}
Figure saved: work/figures/feature_importance.png
Figure saved: work/figures/precision_at_k.png

All exports complete. Paper can now build on these files.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.